# Newton Command Profile Test

`01_newton_single_motor_model.ipynb`에서 구현한 단일 모터 모델을 불러와,
여러 종류의 시험 지령에 대한 Newton 시뮬레이션 응답을 확인한다.

현재 이 파일은

- `01_newton_single_motor_model.ipynb`의 단일 모터 모델 불러오기
- Step / Sine / Chirp / S-curve Command Position 생성
- 사용할 시험 지령 프로파일 선택
- 선택한 Command Position을 이용해 Newton 모터 시뮬레이션 실행
- Viewer를 통한 모터 동작 확인
- Newton의 Feedback Position / Velocity / Torque 응답 확인

정도로 구성되어 있다.

이 파일의 목적은 WMX 데이터와 비교하기 전에,
구현한 단일 모터 모델이 서로 다른 시험 지령에 대해 어떤 응답을 생성하는지 확인하는 것이다.

## Newton Single Motor Model 불러오기

앞서 구현한 `01_newton_single_motor_model.ipynb`의 단일 모터 모델을 현재 노트북으로 불러온다.


In [ ]:
%run 01_newton_single_motor_model.ipynb

## 시험 지령 프로파일 생성 

세 함수 전부 모터에 넣을 Command Position 프로파일을 만들어주는 함수.  
즉, `simulate_motor()`에 넣기 전에 `cmd_pos` 배열을 생성하는 역할

- `profile_step()` → 계단 입력 : overshoot, Ts, 응답속도 보기 좋음    
- `profile_sine()` → 사인파 입력 : 모터가 반복적인 위치 지형을 얼마나 잘 추종하는지, 진폭 감소와 위상 지연 같은 추종 특성을 확인하기 좋음.  
- `profile_chirp()` → 주파수가 변하는 사인파 입력 : 한 번의 시험으로 낮은 주파수부터 높은 주파수까지 훑으면서 주파수에 따른 추종성, 대역폭, 지연 특성을 확인하기 좋음.  
- `profile_scurve()` → 속도 가속도가 부드럽게 변하는 입력 : 가속/감속 구간의 관성·마찰 영향과 실제 운전 추종성을 보기 좋음.  

In [ ]:
def profile_step(
    N,
    dt,
    start_pos,
    target_pos,
    start_time=0.2,
):
    t = np.arange(N) * dt

    # 처음에는 시작 위치 유지
    cmd = np.full(
        N,
        start_pos,
        dtype=np.float32,
    )

    # start_time 이후 목표 위치로 순간 변경
    cmd[t >= start_time] = target_pos

    return cmd


def profile_sine(
    N,
    dt,
    amp,            # 진폭
    freq_hz,        # 주파수
    center=0.0,     # 사인파 중심 위치
):
    t = np.arange(N) * dt

    return (
        center
        + amp * np.sin(2.0 * np.pi * freq_hz * t)
    ).astype(np.float32)


def profile_chirp(
    N,
    dt,
    amp,
    f0,
    f1,
    center=0.0,
):
    t = np.arange(N) * dt   # 시간축 생성
    T = N * dt              # 전체 실험 시간 계산

    freq = f0 + (f1 - f0) * t / T   # 주파수가 시간에 따라 f0 -> f1 로 변화

    phase = (     # 주파수에 맞게 각 순간의 위상도 그에 맞게 누적
        2.0
        * np.pi
        * np.cumsum(freq)
        * dt
    )

    return (      # 누적된 phase로 사인파 Command를 만듦
        center
        + amp * np.sin(phase)
    ).astype(np.float32)


def profile_scurve(
    N,
    dt,
    start_pos,
    target_pos,
    max_velocity,
    max_acceleration,
    max_jerk,
):
    distance = target_pos - start_pos

    if abs(distance) < 1.0e-12:
        return np.full(N, start_pos, dtype=np.float32)

    direction = np.sign(distance)
    D = abs(distance)

    V = abs(max_velocity)
    A = abs(max_acceleration)
    J = abs(max_jerk)

    if V <= 0 or A <= 0 or J <= 0:
        raise ValueError("max_velocity, max_acceleration, max_jerk must be positive.")

    # 최대 가속도에 도달하는 데 필요한 시간
    tj_A = A / J

    # 최대 속도에 먼저 도달하는 경우의 jerk 시간
    tj_V = np.sqrt(V / J)

    # 각 구간의 시간 tj / ta / tv 계산 ---------------------
    if tj_V < tj_A:
        # Amax에 도달하기 전에 Vmax에 도달
        tj = tj_V
        ta = 0.0

        d_to_v = 2.0 * V * tj

        if D >= d_to_v:
            tv = (D - d_to_v) / V
        else:
            # 이동거리가 짧아서 Vmax에도 도달하지 못함
            tj = (D / (2.0 * J)) ** (1.0 / 3.0)
            ta = 0.0
            tv = 0.0

    else:
        # Amax까지 도달 가능
        tj = tj_A

        ta_to_v = V / A - tj
        d_to_v = V * (2.0 * tj + ta_to_v)

        if D >= d_to_v:
            # Amax, Vmax 모두 도달
            ta = ta_to_v
            tv = (D - d_to_v) / V

        else:
            # Vmax에는 도달하지 않음
            ta = (
                -3.0 * tj
                + np.sqrt(tj**2 + 4.0 * D / A)
            ) / 2.0

            if ta < 0.0:
                # Amax에도 도달하지 않음
                tj = (D / (2.0 * J)) ** (1.0 / 3.0)
                ta = 0.0

            tv = 0.0

    # 7개의 S-curve 구간-----------------------------------------------
    durations = np.array([
        tj,     # +J
        ta,     # 일정 가속도
        tj,     # -J
        tv,     # 일정 속도
        tj,     # -J
        ta,     # 일정 감속도
        tj,     # +J
    ])

    jerks = direction * np.array([
        +J,
        0.0,
        -J,
        0.0,
        -J,
        0.0,
        +J,
    ])

    move_time = float(np.sum(durations))

    if N * dt < move_time:
        raise ValueError(
            f"전체 실행시간 {N*dt:.3f}s가 "
            f"S-curve 이동시간 {move_time:.3f}s보다 짧습니다."
        )

    # 각 구간 시작 상태 계산 ------------------------------------------------------------------
    segment_start = []

    q = float(start_pos)
    v = 0.0
    a = 0.0

    for duration, jerk in zip(durations, jerks):

        segment_start.append((q, v, a, jerk))

        q = (
            q
            + v * duration
            + 0.5 * a * duration**2
            + (1.0 / 6.0) * jerk * duration**3
        )

        v = (
            v
            + a * duration
            + 0.5 * jerk * duration**2
        )

        a = a + jerk * duration

    # Command Position 생성 -----------------------------------------------------------
    cmd = np.zeros(N, dtype=np.float32)

    segment_end_times = np.cumsum(durations)
    segment_start_times = np.concatenate(
        ([0.0], segment_end_times[:-1])
    )

    for k in range(N):

        t = k * dt

        # 이동 완료 후에는 목표 위치 유지
        if t >= move_time:
            cmd[k] = target_pos
            continue

        seg = np.searchsorted(
            segment_end_times,
            t,
            side="right",
        )

        q0, v0, a0, jerk = segment_start[seg]
        local_t = t - segment_start_times[seg]

        cmd[k] = (
            q0
            + v0 * local_t
            + 0.5 * a0 * local_t**2
            + (1.0 / 6.0) * jerk * local_t**3
        )

    return cmd

## Step/Sine/chirp/s-curve의 실행함수들

이 셀은 바로 실행하는 것이 아닌, 밑의 과정에서 실행 함수를 선택적으로 사용할 수 있도록  
미리 정의해 두는 과정이다.

In [ ]:
params = MotorParams()

dt = 1.0e-3       # 1 kHz
duration = 7.0     # 모든 시험을 기본 7초
N = int(duration / dt)

def run_step():
    cmd = profile_step(
        N=N,
        dt=dt,

        start_pos=np.deg2rad(30.0),     # 가변 가능
        target_pos=np.deg2rad(90.0),    # 가변 가능
        start_time=1.0,                 # 가변 가능
    )

    return simulate_motor(
        cmd,
        dt,
        params,
        device="cpu",
        use_viewer=True,
    )


def run_sine():
    cmd = profile_sine(
        N=N,
        dt=dt,
        amp=np.deg2rad(45.0),           # 가변 가능
        freq_hz=0.5,                    # 가변 가능
        center=0.0,                     # 가변 가능 [rad]
    )
    return simulate_motor(
        cmd,
        dt,
        params,
        device="cpu",
        use_viewer=True,
    )


def run_chirp():
    cmd = profile_chirp(
        N=N,
        dt=dt,
        amp=np.deg2rad(45.0),           # 가변 가능
        f0=0.2,                         # 가변 가능
        f1=3.0,                         # 가변 가능
        center=0.0,                     # 가변 가능 [rad]
    )
    return simulate_motor(
        cmd,
        dt,
        params,
        device="cpu",
        use_viewer=True,
    )


def run_scurve():
    cmd = profile_scurve(
        N=N,
        dt=dt,

        start_pos=0.0,                          # 가변 가능 [rad]
        target_pos=np.deg2rad(90.0),            # 가변 가능

        max_velocity=np.deg2rad(20.0),          # 가변 가능
        max_acceleration=np.deg2rad(40.0),      # 가변 가능
        max_jerk=np.deg2rad(160.0),             # 가변 가능
    )
    return simulate_motor(
        cmd,
        dt,
        params,
        device="cpu",
        use_viewer=True,
    )

## Step/Sine/Chirp/S-curve Command 중에서 정해서 실행

실제로 N초 짜리 Step/Sine/chirp/S_curve 응답 실험을 진행하는 부분이다.  
일단 step command로 정해서 실행한다. 

In [ ]:
profile_name = "step"  # step/sine/chirp/scurve 중 입력

In [ ]:
if profile_name == "step":
    out = run_step()
    print("Step simulation finished")

elif profile_name == "sine":
    out = run_sine()
    print("Sine simulation finished")

elif profile_name == "chirp":
    out = run_chirp()
    print("Chirp simulation finished")

elif profile_name == "scurve":
    out = run_scurve()
    print("S-curve simulation finished")

else:
    raise ValueError(
        f"Unknown profile: {profile_name}"
    )

## Viewer

선택한 profile에 맞는 simulation 결과를 시각적으로 표시

In [ ]:
# Viewer - 가장 최근에 실행한 simulation 결과 표시

out["viewer"].show_notebook(
    width="100%",
    height=600,
)

## 그래프 출력

Newton 시뮬레이션 안에서 구동되어 feedback된 위치, 속도, 토크에 대한 값을 그래프로 나타낸다.

In [ ]:
# 위치

t = out["time"]

plt.figure(figsize=(10, 4))

plt.plot(
    t,
    out["command_position"],
    label="Command Position",
)

plt.plot(
    t,
    out["feedback_position"],
    label="Sim Feedback Position",
)

plt.xlabel("Time [s]")
plt.ylabel("Position [rad]")
plt.grid()
plt.legend()
plt.show()

# 속도 

plt.figure(figsize=(10, 4))

plt.plot(
    t,
    out["feedback_velocity"],
)

plt.xlabel("Time [s]")
plt.ylabel("Velocity [rad/s]")
plt.grid()
plt.show()


# 토크
plt.figure(figsize=(10, 4))

plt.plot(
    t,
    out["feedback_torque"],
    label="Motor Torque",
)

plt.xlabel("Time [s]")
plt.ylabel("Torque [N·m]")
plt.grid()
plt.legend()
plt.show()